In [ ]:
import os

## Download this Public Dataset from : https://zenodo.org/records/5789064

In [ ]:
#%%
!wget -q https://zenodo.org/records/5789064/files/russellmitchell.zip?download=1 -O russellmitchell.zip

In [ ]:
import zipfile

In [ ]:
with zipfile.ZipFile("russellmitchell.zip", "r") as z:
    names = z.namelist()
    print(len(names), "archivos totales")
    for n in names[:30]:
        print(n)

14911 archivos totales
dataset.yaml
environment/
environment/datasets/
environment/datasets/scenario/
environment/datasets/scenario/processing/
environment/datasets/scenario/processing/process.yaml
environment/datasets/scenario/processing/templates/
environment/datasets/scenario/processing/templates/groups.json.j2
environment/datasets/scenario/processing/templates/rules/
environment/datasets/scenario/processing/templates/rules/dnsmasq.yaml.j2
environment/datasets/scenario/processing/templates/rules/monitoring.yaml.j2
environment/datasets/scenario/processing/templates/rules/0_auth.yaml.j2
environment/datasets/scenario/processing/templates/rules/audit.yaml.j2
environment/datasets/scenario/processing/templates/rules/apache.yaml.j2
environment/datasets/scenario/processing/templates/rules/openvpn.yaml.j2
environment/datasets/scenario/processing/templates/servers.json.j2
environment/datasets/scenario/processing/templates/attacker.yaml.j2
environment/datasets/scenario/processing/templates/att

## Explore Dataset from **RusselMitchell**

In [ ]:
gather_files = [n for n in names if n.startswith("gather/") and not n.endswith("/")]
labels_files = [n for n in names if n.startswith("labels/") and not n.endswith("/")]


In [ ]:
print(len(gather_files), "archivos en gather/")


7477 archivos en gather/


In [ ]:
print(len(labels_files), "archivos en labels/")

8 archivos en labels/


In [ ]:
hosts = sorted(set(n.split("/")[1] for n in gather_files))

In [ ]:
print("Available Hosts:", hosts)

Available Hosts: ['attacker_0', 'cloud_share', 'davey_mail', 'ext_user_0', 'ext_user_1', 'ext_user_2', 'inet-dns', 'inet-firewall', 'internal_employee_0', 'internal_employee_1', 'internal_employee_2', 'internal_employee_3', 'internal_share', 'intranet_server', 'mail', 'monitoring', 'morris_mail', 'remote_employee_0', 'remote_employee_1', 'remote_employee_2', 'vpn', 'webserver']


In [ ]:
#Files Exploration
print("Labeled log files:")
for f in labels_files:
    print(f)

Labeled log files:
labels/inet-firewall/logs/dnsmasq.log
labels/monitoring/logs/logstash/intranet-server/2022-01-24-system.cpu.log
labels/internal_share/logs/audit/audit.log
labels/vpn/logs/openvpn.log
labels/intranet_server/logs/auth.log
labels/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.2
labels/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-error.log.2
labels/intranet_server/logs/audit/audit.log


In [ ]:
target_files = [
    "gather/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.2",
    "gather/intranet_server/logs/auth.log",
    "gather/intranet_server/logs/audit/audit.log",
    "labels/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.2",
    "labels/intranet_server/logs/auth.log",
    "labels/intranet_server/logs/audit/audit.log",
]


In [ ]:
with zipfile.ZipFile("russellmitchell.zip", "r") as z:
    z.extractall("/content/ait", members=target_files)

In [ ]:
base = "/content/ait/gather/intranet_server/logs"

In [ ]:
files_to_check = [
    f"{base}/apache2/intranet.smith.russellmitchell.com-access.log.2",
    f"{base}/auth.log",
    f"{base}/audit/audit.log",
]

In [ ]:
for f in files_to_check:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"{f}: {size_mb:.1f} MB")
    with open(f, "r", errors="ignore") as fh:
        for _ in range(3):
            print(fh.readline().strip())
    print()

/content/ait/gather/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.2: 1.5 MB
10.143.2.91 - - [23/Jan/2022:06:36:13 +0000] "GET / HTTP/1.1" 200 6203 "-" "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:86.0) Gecko/20100101 Firefox/86.0"
10.143.2.91 - - [23/Jan/2022:06:36:14 +0000] "GET /wp-includes/css/dist/block-library/style.min.css?ver=5.8.3 HTTP/1.1" 200 10846 "http://intranet.smith.russellmitchell.com/" "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:86.0) Gecko/20100101 Firefox/86.0"
10.143.2.91 - - [23/Jan/2022:06:36:14 +0000] "GET /wp-includes/js/wp-embed.min.js?ver=5.8.3 HTTP/1.1" 200 1099 "http://intranet.smith.russellmitchell.com/" "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:86.0) Gecko/20100101 Firefox/86.0"

/content/ait/gather/intranet_server/logs/auth.log: 0.0 MB
Jan 23 06:25:05 intranet-server CRON[22883]: pam_unix(cron:session): session closed for user root
Jan 23 06:39:01 intranet-server CRON[23064]: pam_unix(cron:session): session opened for user ro

In [ ]:
import json

In [ ]:
pairs = {
    "apache_access": ("gather/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.2",
                      "labels/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.2"),
    "auth": ("gather/intranet_server/logs/auth.log", "labels/intranet_server/logs/auth.log"),
    "audit": ("gather/intranet_server/logs/audit/audit.log", "labels/intranet_server/logs/audit/audit.log"),
}

In [ ]:

for name, (log_rel, label_rel) in pairs.items():
    with open(f"/content/ait/{log_rel}", "r", errors="ignore") as f:
        total_lines = sum(1 for _ in f)
    with open(f"/content/ait/{label_rel}", "r") as f:
        labeled_lines = sum(1 for _ in f)
    print(f"{name}: {total_lines} total lines, {labeled_lines} labeled as attack")

apache_access: 8530 total lines, 7695 labeled as attack
auth: 272 total lines, 8 labeled as attack
audit: 2316 total lines, 9 labeled as attack


In [ ]:
with zipfile.ZipFile("russellmitchell.zip", "r") as z:
    infolist = z.infolist()

In [ ]:
apache_access = [
    (i.filename, i.file_size / (1024 * 1024))
    for i in infolist
    if i.filename.startswith("gather/")
    and "apache2" in i.filename
    and "access" in i.filename
    and not i.is_dir()
]

In [ ]:
for name, size_mb in apache_access:
    print(f"{name}: {size_mb:.1f} MB")

gather/webserver/logs/apache2/mail.smith.russellmitchell.com-access.log.2: 2.5 MB
gather/webserver/logs/apache2/proxy-access.log.4: 0.1 MB
gather/webserver/logs/apache2/proxy-access.log: 0.0 MB
gather/webserver/logs/apache2/cloud.smith.russellmitchell.com-access.log: 0.0 MB
gather/webserver/logs/apache2/cloud.smith.russellmitchell.com-access.log.1: 1.2 MB
gather/webserver/logs/apache2/proxy-access.log.2: 0.1 MB
gather/webserver/logs/apache2/cloud.smith.russellmitchell.com-access.log.3: 1.8 MB
gather/webserver/logs/apache2/mail.smith.russellmitchell.com-access.log: 0.0 MB
gather/webserver/logs/apache2/cloud.smith.russellmitchell.com-access.log.4: 1.7 MB
gather/webserver/logs/apache2/access.log: 0.0 MB
gather/webserver/logs/apache2/mail.smith.russellmitchell.com-access.log.4: 2.3 MB
gather/webserver/logs/apache2/mail.smith.russellmitchell.com-access.log.1: 2.6 MB
gather/webserver/logs/apache2/proxy-access.log.1: 0.1 MB
gather/webserver/logs/apache2/cloud.smith.russellmitchell.com-access.

## EDA v1

In [ ]:
import re
from datetime import datetime

In [ ]:
extra_files = [
    "gather/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.1",
    "gather/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.3",
    "gather/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.4",
]


In [ ]:
with zipfile.ZipFile("russellmitchell.zip", "r") as z:
    z.extractall("/content/ait", members=extra_files)

In [ ]:
ts_re = re.compile(r"\[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2}) ")

In [ ]:
def time_range(path):
    stamps = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            m = ts_re.search(line)
            if m:
                stamps.append(datetime.strptime(m.group(1), "%d/%b/%Y:%H:%M:%S"))
    return len(stamps), min(stamps), max(stamps)


In [ ]:
all_files = extra_files + ["gather/intranet_server/logs/apache2/intranet.smith.russellmitchell.com-access.log.2"]

In [ ]:
for rel in all_files:
    n, first, last = time_range(f"/content/ait/{rel}")
    print(f"{rel.split('/')[-1]}: {n} lines, {first} -> {last}")

intranet.smith.russellmitchell.com-access.log.1: 601 lines, 2022-01-24 07:34:57 -> 2022-01-24 20:59:10
intranet.smith.russellmitchell.com-access.log.3: 933 lines, 2022-01-22 07:26:47 -> 2022-01-22 20:12:45
intranet.smith.russellmitchell.com-access.log.4: 1120 lines, 2022-01-21 05:55:22 -> 2022-01-22 06:16:08
intranet.smith.russellmitchell.com-access.log.2: 8530 lines, 2022-01-23 06:36:13 -> 2022-01-24 04:37:25


In [ ]:
import os
import json
import re
import pandas as pd

In [ ]:
LOG_RE = re.compile(
    r'^(?P<ip>\S+) \S+ \S+ \[(?P<ts>[^\]]+)\] "(?P<request>.*?)" '
    r'(?P<status>\d{3}) (?P<size>\S+) "(?P<referer>.*?)" "(?P<ua>.*?)"\s*$'
)

In [ ]:
BASE = "/content/ait"
LOG_DIR = "gather/intranet_server/logs/apache2"
LABEL_DIR = "labels/intranet_server/logs/apache2"
PREFIX = "intranet.smith.russellmitchell.com-access.log"


In [ ]:
def load_labels(path):
    labeled = {}
    with open(path, "r") as f:
        for raw in f:
            obj = json.loads(raw)
            labeled[obj["line"]] = "|".join(obj["labels"])
    return labeled

In [ ]:
def parse_log(path, labels):
    rows = []
    unmatched = 0
    with open(path, "r", errors="ignore") as f:
        for lineno, raw in enumerate(f, start=1):
            m = LOG_RE.match(raw.rstrip("\n"))
            if not m:
                unmatched += 1
                continue
            row = m.groupdict()
            row["lineno"] = lineno
            row["attack_labels"] = labels.get(lineno, "")
            row["target"] = int(lineno in labels)
            rows.append(row)
    return rows, unmatched

In [ ]:
frames = []
for suffix in ["1", "2", "3", "4"]:
    log_path = f"{BASE}/{LOG_DIR}/{PREFIX}.{suffix}"
    label_path = f"{BASE}/{LABEL_DIR}/{PREFIX}.{suffix}"
    labels = load_labels(label_path) if os.path.exists(label_path) else {}
    rows, unmatched = parse_log(log_path, labels)
    part = pd.DataFrame(rows)
    part["source_file"] = f"log.{suffix}"
    frames.append(part)
    print(f"log.{suffix}: parsed {len(part)}, unmatched {unmatched}, attack {part['target'].sum()}")


log.1: parsed 601, unmatched 0, attack 0
log.2: parsed 8530, unmatched 0, attack 7695
log.3: parsed 933, unmatched 0, attack 0
log.4: parsed 1120, unmatched 0, attack 0


In [ ]:
ait_df = pd.concat(frames, ignore_index=True)
print(ait_df.shape)
print(ait_df["target"].value_counts())

(11184, 11)
target
1    7695
0    3489
Name: count, dtype: int64


In [ ]:
print("Unique IPs by class:")
print(ait_df.groupby("target")["ip"].nunique())

Unique IPs by class:
target
0    6
1    1
Name: ip, dtype: int64


In [ ]:
print("Top IPs for attack class:")
print(ait_df[ait_df["target"] == 1]["ip"].value_counts().head(5))

Top IPs for attack class:
ip
172.19.131.174    7695
Name: count, dtype: int64


In [ ]:
print("Top user agents for attack class:")
print(ait_df[ait_df["target"] == 1]["ua"].value_counts().head(5))

Top user agents for attack class:
ua
Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.1)                                 4462
WPScan v3.8.20 (https://wpscan.com/wordpress-security-scanner)                     3186
python-requests/2.27.1                                                               35
-                                                                                     6
Mozilla/5.0 (compatible; Nmap Scripting Engine; https://nmap.org/book/nse.html)       6
Name: count, dtype: int64


In [ ]:
print("Status codes by class (share):")
print(pd.crosstab(ait_df["target"], ait_df["status"], normalize="index").round(3))

Status codes by class (share):
status    200    301  302    304    403    404  405    408  500
target                                                         
0       0.955  0.000  0.0  0.003  0.000  0.026  0.0  0.015  0.0
1       0.008  0.001  0.0  0.000  0.001  0.990  0.0  0.000  0.0


In [ ]:
print("Attack steps:")
print(ait_df[ait_df["target"] == 1]["attack_labels"].value_counts())

Attack steps:
attack_labels
attacker_http|foothold|dirb               4462
attacker_http|foothold|wpscan             3186
attacker_http|foothold|webshell_cmd         28
attacker_http|foothold|service_scan          8
service_scan|foothold                        4
webshell_cmd|escalate                        4
attacker_http|foothold|webshell_upload       3
Name: count, dtype: int64


In [ ]:
print("Status codes by class (share):")
print(pd.crosstab(ait_df["target"], ait_df["status"], normalize="index").round(3))

Status codes by class (share):
status    200    301  302    304    403    404  405    408  500
target                                                         
0       0.955  0.000  0.0  0.003  0.000  0.026  0.0  0.015  0.0
1       0.008  0.001  0.0  0.000  0.001  0.990  0.0  0.000  0.0


In [ ]:
!ls -lah /content/ait

total 16K
drwxr-xr-x 4 root root 4.0K Oct  1 03:57 .
drwxr-xr-x 1 root root 4.0K Oct  1 03:57 ..
drwxr-xr-x 3 root root 4.0K Oct  1 03:57 gather
drwxr-xr-x 3 root root 4.0K Oct  1 03:57 labels


In [ ]:
WINDOW_SECONDS = 60

In [ ]:
ait_df["ts_parsed"] = pd.to_datetime(
    ait_df["ts"], format="%d/%b/%Y:%H:%M:%S %z", utc=True
)

In [ ]:
epoch_zero = pd.Timestamp("1970-01-01", tz="UTC")
ait_df["epoch"] = (ait_df["ts_parsed"] - epoch_zero).dt.total_seconds()
ait_df["window"] = (ait_df["epoch"] // WINDOW_SECONDS).astype(int)


In [ ]:
req_parts = ait_df["request"].str.split(" ", n=2, expand=True)
ait_df["method"] = req_parts[0]
ait_df["path"] = req_parts[1].fillna("")

In [ ]:
ait_df["status"] = ait_df["status"].astype(int)
ait_df["is_404"] = (ait_df["status"] == 404).astype(int)
ait_df["is_2xx"] = ait_df["status"].between(200, 299).astype(int)
ait_df["has_query"] = ait_df["path"].str.contains(r"\?", regex=True).astype(int)
ait_df["is_post"] = (ait_df["method"] == "POST").astype(int)
ait_df["path_len"] = ait_df["path"].str.len()

In [ ]:
win_df = (
    ait_df.groupby(["ip", "window"])
    .agg(
        req_count=("lineno", "size"),
        ratio_404=("is_404", "mean"),
        ratio_2xx=("is_2xx", "mean"),
        unique_paths=("path", "nunique"),
        ua_count=("ua", "nunique"),
        query_ratio=("has_query", "mean"),
        post_ratio=("is_post", "mean"),
        mean_path_len=("path_len", "mean"),
        attack_lines=("target", "sum"),
        target=("target", "max"),
        start_epoch=("epoch", "min"),
    )
    .reset_index()
)

In [ ]:
win_df["unique_path_ratio"] = win_df["unique_paths"] / win_df["req_count"]

In [ ]:
print(win_df.shape)

(820, 14)


In [ ]:
print(win_df["target"].value_counts())


target
0    816
1      4
Name: count, dtype: int64


In [ ]:
print(win_df.groupby("target")["req_count"].describe())

        count         mean          std  min   25%     50%      75%     max
target                                                                     
0       816.0     4.275735     5.579647  1.0   1.0     2.0     4.00    21.0
1         4.0  1923.750000  2317.827773  2.0  11.0  1503.0  3415.75  4687.0


In [ ]:
mixed = ((win_df["attack_lines"] > 0) & (win_df["attack_lines"] < win_df["req_count"])).sum()


In [ ]:
print("Mixed windows:", mixed)

Mixed windows: 0


## Disregard the Solo RusselMitchell dataset is not enough to train a reliable ML Model

In [ ]:
!pip install -q remotezip

In [ ]:
from remotezip import RemoteZip

In [ ]:
url = "https://zenodo.org/records/5789064/files/santos.zip?download=1"

In [ ]:
with RemoteZip(url) as z:
    names = z.namelist()
    print(len(names), "entries in remote zip")
    for n in names:
        if n.startswith("labels/") and "apache2" in n:
            print(n)

14908 entries in remote zip
labels/intranet_server/logs/apache2/
labels/intranet_server/logs/apache2/intranet.smith.santos.com-access.log
labels/intranet_server/logs/apache2/intranet.smith.santos.com-error.log


In [ ]:
from remotezip import RemoteZip

ZENODO = "https://zenodo.org/records/5789064/files/{}.zip?download=1"

In [ ]:
# Download the other Model by testbed
def fetch_testbed(name, out_root="/content/ait_multi"):
    url = ZENODO.format(name)
    with RemoteZip(url) as z:
        wanted = [
            i for i in z.infolist()
            if not i.is_dir()
            and "intranet_server/logs/apache2/" in i.filename
            and "-access.log" in i.filename
            and i.filename.startswith(("gather/", "labels/"))
        ]
        for info in wanted:
            print(f"{info.filename}: {info.file_size / 1024:.0f} KB")
            z.extract(info, f"{out_root}/{name}")
    return [i.filename for i in wanted]


In [ ]:
santos_files = fetch_testbed("santos")

gather/intranet_server/logs/apache2/intranet.smith.santos.com-access.log.2: 424 KB
gather/intranet_server/logs/apache2/intranet.smith.santos.com-access.log: 1731 KB
gather/intranet_server/logs/apache2/intranet.smith.santos.com-access.log.1: 562 KB
gather/intranet_server/logs/apache2/intranet.smith.santos.com-access.log.3: 618 KB
labels/intranet_server/logs/apache2/intranet.smith.santos.com-access.log: 1620 KB


In [ ]:
print(len(santos_files), "files extracted")

5 files extracted


In [ ]:
TESTBEDS = ["fox", "harrison", "shaw", "wardbeck", "wheeler", "wilson"]

In [ ]:
for tb in TESTBEDS:
    try:
        files = fetch_testbed(tb)
        print(f"{tb}: {len(files)} files")
    except Exception as e:
        print(f"{tb}: FAILED ({e})")

gather/intranet_server/logs/apache2/intranet.price.fox.org-access.log.5: 709 KB
gather/intranet_server/logs/apache2/intranet.price.fox.org-access.log.4: 534 KB
gather/intranet_server/logs/apache2/intranet.price.fox.org-access.log.1: 510 KB
gather/intranet_server/logs/apache2/intranet.price.fox.org-access.log.2: 65663 KB
gather/intranet_server/logs/apache2/intranet.price.fox.org-access.log.3: 903 KB
labels/intranet_server/logs/apache2/intranet.price.fox.org-access.log.2: 85086 KB
fox: 6 files
gather/intranet_server/logs/apache2/intranet.mannsmith.harrison.com-access.log.3: 1443 KB
gather/intranet_server/logs/apache2/intranet.mannsmith.harrison.com-access.log.5: 1457 KB
gather/intranet_server/logs/apache2/intranet.mannsmith.harrison.com-access.log.1: 66617 KB
gather/intranet_server/logs/apache2/intranet.mannsmith.harrison.com-access.log.2: 1149 KB
gather/intranet_server/logs/apache2/intranet.mannsmith.harrison.com-access.log.4: 1371 KB
labels/intranet_server/logs/apache2/intranet.mannsmi

In [ ]:
ROOT = "/content/ait_multi"

In [ ]:
def count_lines(path):
    with open(path, "r", errors="ignore") as f:
        return sum(1 for _ in f)

In [ ]:
def count_lines(path):
    with open(path, "r", errors="ignore") as f:
        return sum(1 for _ in f)


In [ ]:
for tb in ["santos", "fox", "harrison", "shaw", "wardbeck", "wheeler", "wilson"]:
    label_dir = f"{ROOT}/{tb}/labels/intranet_server/logs/apache2"
    for fname in sorted(os.listdir(label_dir)):
        labeled = count_lines(f"{label_dir}/{fname}")
        total = count_lines(f"{ROOT}/{tb}/gather/intranet_server/logs/apache2/{fname}")
        print(f"{tb}/{fname}: {labeled} attack of {total} lines")

santos/intranet.smith.santos.com-access.log: 7794 attack of 9462 lines
fox/intranet.price.fox.org-access.log.2: 410841 attack of 412922 lines
harrison/intranet.mannsmith.harrison.com-access.log.1: 415376 attack of 420831 lines
shaw/intranet.thomasmurray.shaw.info-access.log.2: 5226 attack of 6626 lines
wardbeck/intranet.hurstwong.wardbeck.info-access.log.1: 5299 attack of 11262 lines
wheeler/intranet.flores.wheeler.biz-access.log.1: 431560 attack of 433194 lines
wilson/intranet.hallbrown.wilson.com-access.log.2: 428116 attack of 434598 lines


In [ ]:
import os
import json
import re
import pandas as pd

In [ ]:
WINDOW_SECONDS = 10
EPOCH_ZERO = pd.Timestamp("1970-01-01", tz="UTC")

In [ ]:
LOG_RE = re.compile(
    r'^(?P<ip>\S+) \S+ \S+ \[(?P<ts>[^\]]+)\] "(?P<request>.*?)" '
    r'(?P<status>\d{3}) (?P<size>\S+) "(?P<referer>.*?)" "(?P<ua>.*?)"\s*$'
)

In [ ]:
SIM_RANGES = {
    "russellmitchell": ("2022-01-21", "2022-01-25"),
    "santos": ("2022-01-14", "2022-01-18"),
    "fox": ("2022-01-15", "2022-01-20"),
    "harrison": ("2022-02-04", "2022-02-09"),
    "shaw": ("2022-01-25", "2022-01-31"),
    "wardbeck": ("2022-01-19", "2022-01-24"),
    "wheeler": ("2022-01-26", "2022-01-31"),
    "wilson": ("2022-02-03", "2022-02-09"),
}


In [ ]:
def testbed_dirs(name):
    root = "/content/ait" if name == "russellmitchell" else f"/content/ait_multi/{name}"
    return (
        f"{root}/gather/intranet_server/logs/apache2",
        f"{root}/labels/intranet_server/logs/apache2",
    )



In [ ]:

def load_label_lines(path):
    lines = set()
    with open(path, "r") as f:
        for raw in f:
            lines.add(json.loads(raw)["line"])
    return lines

In [ ]:
def parse_file(path, labeled_lines):
    rows = []
    unmatched = 0
    with open(path, "r", errors="ignore") as f:
        for lineno, raw in enumerate(f, start=1):
            m = LOG_RE.match(raw.rstrip("\n"))
            if m is None:
                unmatched += 1
                continue
            rows.append((m["ip"], m["ts"], m["request"], m["status"], m["ua"],
                         int(lineno in labeled_lines)))

    cols = ["ip", "ts", "request", "status", "ua", "target"]
    return pd.DataFrame(rows, columns=cols), unmatched


In [ ]:
def to_windows(df, sim_start, sim_end):
    df["ts_parsed"] = pd.to_datetime(df["ts"], format="%d/%b/%Y:%H:%M:%S %z", utc=True)
    start = pd.Timestamp(sim_start, tz="UTC")
    end = pd.Timestamp(sim_end, tz="UTC")
    df = df[(df["ts_parsed"] >= start) & (df["ts_parsed"] < end)].copy()
    df["epoch"] = (df["ts_parsed"] - EPOCH_ZERO).dt.total_seconds()
    df["window"] = (df["epoch"] // WINDOW_SECONDS).astype(int)

    parts = df["request"].str.split(" ", n=2, expand=True)
    df["path"] = parts[1].fillna("")
    df["is_post"] = (parts[0] == "POST").astype(int)
    df["status"] = df["status"].astype(int)
    df["is_404"] = (df["status"] == 404).astype(int)
    df["is_2xx"] = df["status"].between(200, 299).astype(int)
    df["has_query"] = df["path"].str.contains("?", regex=False).astype(int)
    df["path_len"] = df["path"].str.len()

    win = (
        df.groupby(["ip", "window"])
        .agg(
            req_count=("path", "size"),
            ratio_404=("is_404", "mean"),
            ratio_2xx=("is_2xx", "mean"),
            unique_paths=("path", "nunique"),
            ua_count=("ua", "nunique"),
            query_ratio=("has_query", "mean"),
            post_ratio=("is_post", "mean"),
            mean_path_len=("path_len", "mean"),
            target=("target", "max"),
        )
        .reset_index()
    )
    win["unique_path_ratio"] = win["unique_paths"] / win["req_count"]
    return win

In [ ]:
## Having Problem during Data Exploration

In [ ]:
all_windows = []
for tb, (sim_start, sim_end) in SIM_RANGES.items():
    gather_dir, label_dir = testbed_dirs(tb)
    for fname in sorted(os.listdir(gather_dir)):
        if "-access.log" not in fname:
            continue
        label_path = f"{label_dir}/{fname}"
        labeled = load_label_lines(label_path) if os.path.exists(label_path) else set()
        df, unmatched = parse_file(f"{gather_dir}/{fname}", labeled)
        parsed = len(df)
        win = to_windows(df, sim_start, sim_end)
        win["testbed"] = tb
        win["source_file"] = fname
        all_windows.append(win)
        print(f"{tb}/{fname}: parsed {parsed}, unmatched {unmatched}, "
              f"windows {len(win)}, attack windows {int(win['target'].sum())}")


russellmitchell/intranet.smith.russellmitchell.com-access.log.1: parsed 601, unmatched 0, windows 192, attack windows 0
russellmitchell/intranet.smith.russellmitchell.com-access.log.2: parsed 8530, unmatched 0, windows 237, attack windows 15
russellmitchell/intranet.smith.russellmitchell.com-access.log.3: parsed 933, unmatched 0, windows 236, attack windows 0
russellmitchell/intranet.smith.russellmitchell.com-access.log.4: parsed 1120, unmatched 0, windows 327, attack windows 0
santos/intranet.smith.santos.com-access.log: parsed 9462, unmatched 0, windows 499, attack windows 18
santos/intranet.smith.santos.com-access.log.1: parsed 2228, unmatched 0, windows 655, attack windows 0
santos/intranet.smith.santos.com-access.log.2: parsed 1706, unmatched 0, windows 696, attack windows 0
santos/intranet.smith.santos.com-access.log.3: parsed 2388, unmatched 0, windows 564, attack windows 0
fox/intranet.price.fox.org-access.log.1: parsed 1911, unmatched 0, windows 524, attack windows 0
fox/intra

In [ ]:
## CrackDown  ETL

In [ ]:
win_df = pd.concat(all_windows, ignore_index=True)


In [ ]:
print(win_df.shape)


(43024, 14)


## Further Audit Datasets

In [ ]:
print(win_df.groupby("testbed")["target"].agg(["size", "sum"]))

                  size  sum
testbed                    
fox               3753  126
harrison          6117  172
russellmitchell    992   15
santos            2414   18
shaw              3213    9
wardbeck         14669   19
wheeler           2393  108
wilson            9473  130


In [ ]:
summary = (
    win_df.groupby(["testbed", "target"])
    .agg(
        windows=("req_count", "size"),
        req_min=("req_count", "min"),
        req_median=("req_count", "median"),
        req_max=("req_count", "max"),
        ratio_404_median=("ratio_404", "median"),
        uniq_ratio_median=("unique_path_ratio", "median"),
    )
    .round(2)
)

In [ ]:
print(summary)

                        windows  req_min  req_median  req_max  \
testbed         target                                          
fox             0          3627        1         1.0       91   
                1           126        1      3498.0     3913   
harrison        0          5945        1         1.0       96   
                1           172        1      2621.0     2917   
russellmitchell 0           977        1         1.0       19   
                1            15        1         6.0     3249   
santos          0          2396        1         1.0       77   
                1            18        1         4.0     3065   
shaw            0          3204        1         1.0       91   
                1             9        1         4.0     2383   
wardbeck        0         14650        1         1.0       91   
                1            19        1         3.0     3457   
wheeler         0          2285        1         2.0       42   
                1        

In [ ]:
from xgboost import XGBClassifier

In [ ]:
FEATURES = [
    "req_count", "ratio_404", "ratio_2xx", "unique_paths", "ua_count",
    "query_ratio", "post_ratio", "mean_path_len", "unique_path_ratio",
]

In [ ]:
RULE_THRESHOLD = 100

In [ ]:
rows = []
for held_out in sorted(win_df["testbed"].unique()):
    train = win_df[win_df["testbed"] != held_out]
    test = win_df[win_df["testbed"] == held_out]

    pos_weight = (train["target"] == 0).sum() / (train["target"] == 1).sum()
    model = XGBClassifier(
        n_estimators=200,
        max_depth=3,
        scale_pos_weight=pos_weight,
        eval_metric="logloss",
        random_state=42,
    )
    model.fit(train[FEATURES], train["target"])

    pred_model = model.predict(test[FEATURES])
    pred_rule = (test["req_count"] >= RULE_THRESHOLD).astype(int).to_numpy()
    y = test["target"].to_numpy()

    rows.append({
        "held_out": held_out,
        "attack_windows": int(y.sum()),
        "model_caught": int(((pred_model == 1) & (y == 1)).sum()),
        "model_false_pos": int(((pred_model == 1) & (y == 0)).sum()),
        "rule_caught": int(((pred_rule == 1) & (y == 1)).sum()),
        "rule_false_pos": int(((pred_rule == 1) & (y == 0)).sum()),
    })


In [ ]:
result = pd.DataFrame(rows)

In [ ]:
print(result.to_string(index=False))


       held_out  attack_windows  model_caught  model_false_pos  rule_caught  rule_false_pos
            fox             126           124                3          119               0
       harrison             172           170                9          162               0
russellmitchell              15            14                1            5               0
         santos              18            17                3            4               0
           shaw               9             7                4            4               0
       wardbeck              19            17               10            4               0
        wheeler             108           107                3          102               0
         wilson             130           129               18          123               0


In [ ]:
print(result.drop(columns="held_out").sum())

attack_windows     597
model_caught       585
model_false_pos     51
rule_caught        523
rule_false_pos       0
dtype: int64


In [ ]:
## Unreliable model to be a Sensor

In [ ]:
win_df = win_df.reset_index(drop=True)
win_df["pred_loto"] = 0

In [ ]:
## Try a different Approach

In [ ]:
for held_out in sorted(win_df["testbed"].unique()):
    tr = win_df.index[win_df["testbed"] != held_out]
    te = win_df.index[win_df["testbed"] == held_out]
    pos_weight = (win_df.loc[tr, "target"] == 0).sum() / (win_df.loc[tr, "target"] == 1).sum()
    m = XGBClassifier(
        n_estimators=200, max_depth=3, scale_pos_weight=pos_weight,
        eval_metric="logloss", random_state=42,
    )
    m.fit(win_df.loc[tr, FEATURES], win_df.loc[tr, "target"])
    win_df.loc[te, "pred_loto"] = m.predict(win_df.loc[te, FEATURES])

In [ ]:
## Check New Metrics

In [ ]:
small_attack = win_df[(win_df["target"] == 1) & (win_df["req_count"] < RULE_THRESHOLD)]
benign = win_df[win_df["target"] == 0]
print(len(small_attack), "attack windows with req_count < 100")
print(pd.DataFrame({
    "small_attack_median": small_attack[FEATURES].median(),
    "benign_median": benign[FEATURES].median(),
}).round(2))

74 attack windows with req_count < 100
                   small_attack_median  benign_median
req_count                          3.0            1.0
ratio_404                          0.0            0.0
ratio_2xx                          1.0            1.0
unique_paths                       3.0            1.0
ua_count                           1.0            1.0
query_ratio                        1.0            0.0
post_ratio                         0.0            1.0
mean_path_len                     88.0           24.0
unique_path_ratio                  1.0            1.0


In [ ]:
missed = win_df[(win_df["target"] == 1) & (win_df["pred_loto"] == 0)]

In [ ]:
print("Missed attack windows:")
print(missed[["testbed"] + FEATURES].round(2).to_string())

Missed attack windows:
               testbed  req_count  ratio_404  ratio_2xx  unique_paths  ua_count  query_ratio  post_ratio  mean_path_len  unique_path_ratio
384    russellmitchell          2       0.00       1.00             1         1         0.00        0.00           1.00                0.5
1071            santos          1       0.00       1.00             1         1         0.00        0.00           1.00                1.0
4361               fox          1       0.00       1.00             1         1         0.00        0.00           1.00                1.0
4482               fox          7       0.00       1.00             7         1         0.71        0.14          55.71                1.0
7160          harrison          1       0.00       1.00             1         1         0.00        0.00           1.00                1.0
7326          harrison          5       0.00       1.00             5         1         0.60        0.20          42.40                1.0
1392

In [ ]:
## Having Problems with  Logs

In [ ]:
pd.set_option("display.max_colwidth", 120)

print("Top benign paths:")
print(ait_df[ait_df["target"] == 0]["path"].value_counts().head(8))

Top benign paths:
path
/wp-admin/admin-ajax.php                                                            475
/                                                                                   318
/wp-content/themes/go/dist/css/style-shared.min.css?ver=1.5.2                       191
/wp-includes/css/dist/block-library/style.min.css?ver=5.8.3                         180
/wp-content/themes/go/dist/css/design-styles/style-traditional.min.css?ver=1.5.2    178
/wp-content/themes/go/dist/js/frontend.min.js?ver=1.5.2                             177
/wp-includes/js/wp-emoji-release.min.js?ver=5.8.3                                   165
/wp-includes/js/wp-embed.min.js?ver=5.8.3                                           163
Name: count, dtype: int64


In [ ]:
non_scan = ait_df[
    (ait_df["target"] == 1)
    & (~ait_df["attack_labels"].str.contains("dirb|wpscan"))
]

In [ ]:

print("Attack requests that are not dirb or wpscan:")
print(non_scan[["method", "path", "status"]].head(12).to_string())

Attack requests that are not dirb or wpscan:
     method                       path  status
1432    GET                          /     200
1435    GET                          /     200
1436   POST                       /sdk     404
1437    GET  /nmaplowercheck1642996621     404
1438    GET  /nmaplowercheck1642996621     404
1439   POST                       /sdk     404
1440    GET                          /     200
1441    GET                          /     200
1442    GET                     /HNAP1     404
1443    GET                     /HNAP1     404
1444    GET                          /     200
1445    GET                          /     200


In [ ]:
## Check the webshell requests logs

In [ ]:
#%%
webshell = ait_df[ait_df["attack_labels"].str.contains("webshell")]
print(len(webshell), "webshell requests")
print(webshell[["method", "path", "status"]].head(15).to_string())

35 webshell requests
     method                                                                                                     path  status
9095    GET                                                                                                        /     200
9096    GET                                                                                                    /?p=5     200
9097   POST                                                                                 /wp-admin/admin-ajax.php     200
9098    GET                  /wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php?wp_meta=WyJ3aG9hbWkiXQ%3D%3D     200
9099    GET                  /wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php?wp_meta=WyJ1bmFtZSIsICItciJd     200
9100    GET    /wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php?wp_meta=WyJjYXQiLCAiL2V0Yy9wcm9maWxlIl0%3D     200
9101    GET                      /wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php?wp_meta=WyJ3

In [ ]:
TRANSFER_FEATURES = [
    "req_count", "ratio_404", "ratio_2xx",
    "unique_paths", "ua_count", "unique_path_ratio",
]


In [ ]:
def loto_eval(features):
    out = {}
    for held_out in sorted(win_df["testbed"].unique()):
        tr = win_df["testbed"] != held_out
        te = ~tr
        pos_weight = (win_df.loc[tr, "target"] == 0).sum() / (win_df.loc[tr, "target"] == 1).sum()
        m = XGBClassifier(
            n_estimators=200, max_depth=3, scale_pos_weight=pos_weight,
            eval_metric="logloss", random_state=42,
        )
        m.fit(win_df.loc[tr, features], win_df.loc[tr, "target"])
        pred = m.predict(win_df.loc[te, features])
        y = win_df.loc[te, "target"].to_numpy()
        out[held_out] = (int(((pred == 1) & (y == 1)).sum()), int(((pred == 1) & (y == 0)).sum()))
    return out

In [ ]:
full = loto_eval(FEATURES)
transfer = loto_eval(TRANSFER_FEATURES)

In [ ]:
rows = []
for tb in full:
    rows.append({
        "held_out": tb,
        "attack_windows": int(win_df.loc[win_df["testbed"] == tb, "target"].sum()),
        "full_caught": full[tb][0], "full_fp": full[tb][1],
        "transfer_caught": transfer[tb][0], "transfer_fp": transfer[tb][1],
    })

In [ ]:

cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))
print(cmp_df.drop(columns="held_out").sum())

       held_out  attack_windows  full_caught  full_fp  transfer_caught  transfer_fp
            fox             126          124        3              122           49
       harrison             172          170        9              169           82
russellmitchell              15           14        1               11           13
         santos              18           17        3               14           33
           shaw               9            7        4                7           65
       wardbeck              19           17       10               15           82
        wheeler             108          107        3              108           47
         wilson             130          129       18              127          172
attack_windows     597
full_caught        585
full_fp             51
transfer_caught    573
transfer_fp        543
dtype: int64


In [ ]:
## Look for a better Threshold
oof = pd.Series(0.0, index=win_df.index)

for held_out in sorted(win_df["testbed"].unique()):
    tr = win_df["testbed"] != held_out
    te = ~tr
    pos_weight = (win_df.loc[tr, "target"] == 0).sum() / (win_df.loc[tr, "target"] == 1).sum()
    m = XGBClassifier(
        n_estimators=200, max_depth=3, scale_pos_weight=pos_weight,
        eval_metric="logloss", random_state=42,
    )
    m.fit(win_df.loc[tr, TRANSFER_FEATURES], win_df.loc[tr, "target"])
    oof.loc[te] = m.predict_proba(win_df.loc[te, TRANSFER_FEATURES])[:, 1]

y_all = win_df["target"]
total_attack = int(y_all.sum())
for thr in [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]:
    pred = oof >= thr
    caught = int((pred & (y_all == 1)).sum())
    false_pos = int((pred & (y_all == 0)).sum())
    print(f"threshold {thr}: caught {caught} of {total_attack}, false positives {false_pos}")

threshold 0.5: caught 573 of 597, false positives 543
threshold 0.7: caught 573 of 597, false positives 453
threshold 0.8: caught 554 of 597, false positives 336
threshold 0.9: caught 537 of 597, false positives 107
threshold 0.95: caught 535 of 597, false positives 0
threshold 0.99: caught 535 of 597, false positives 0


In [ ]:
import json
import xgboost
from xgboost import XGBClassifier

In [ ]:
FINAL_THRESHOLD = 0.95

In [ ]:
pos_weight = (win_df["target"] == 0).sum() / (win_df["target"] == 1).sum()

In [ ]:
final_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=42,
)

In [ ]:
final_model.fit(win_df[TRANSFER_FEATURES], win_df["target"])

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
## Train the best ever model after hours of Data Exploration

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

In [ ]:
proba_in = final_model.predict_proba(win_df[TRANSFER_FEATURES])[:, 1]
pred_in = (proba_in >= FINAL_THRESHOLD).astype(int)

In [ ]:
print("In-sample (rows the model trained on, optimistic):")
print(classification_report(win_df["target"], pred_in, digits=4))

In-sample (rows the model trained on, optimistic):
              precision    recall  f1-score   support

           0     0.9986    1.0000    0.9993     42427
           1     1.0000    0.8995    0.9471       597

    accuracy                         0.9986     43024
   macro avg     0.9993    0.9497    0.9732     43024
weighted avg     0.9986    0.9986    0.9986     43024



In [ ]:
pred_oof = (oof >= FINAL_THRESHOLD).astype(int)

In [ ]:
print("Out-of-fold (each testbed predicted by a model that never saw it):")
print(classification_report(win_df["target"], pred_oof, digits=4))
print("Out-of-fold ROC AUC:", roc_auc_score(win_df["target"], oof))

Out-of-fold (each testbed predicted by a model that never saw it):
              precision    recall  f1-score   support

           0     0.9985    1.0000    0.9993     42427
           1     1.0000    0.8961    0.9452       597

    accuracy                         0.9986     43024
   macro avg     0.9993    0.9481    0.9722     43024
weighted avg     0.9986    0.9986    0.9985     43024

Out-of-fold ROC AUC: 0.9832050076831153


In [ ]:
importances = pd.Series(
    final_model.feature_importances_, index=TRANSFER_FEATURES
).sort_values(ascending=False)

In [ ]:
print(importances.round(4))

ratio_404            0.8705
unique_paths         0.0727
req_count            0.0440
ratio_2xx            0.0068
ua_count             0.0049
unique_path_ratio    0.0011
dtype: float32


In [ ]:
import joblib

In [ ]:
joblib.dump(final_model, "recon_sensor_sgb.joblib")

['recon_sensor_sgb.joblib']

In [ ]:
edge_cases = pd.DataFrame(
    [
        ("1 request, 404", 1, 1.0, 0.0, 1, 1, 1.0),
        ("3 requests, all 404, distinct paths", 3, 1.0, 0.0, 3, 1, 1.0),
        ("20 requests, all 404, distinct paths", 20, 1.0, 0.0, 20, 1, 1.0),
        ("20 requests, half 404", 20, 0.5, 0.5, 20, 1, 1.0),
        ("1 request, 200", 1, 0.0, 1.0, 1, 1, 1.0),
        ("500 requests, all 200 (crawler)", 500, 0.0, 1.0, 500, 1, 1.0),
    ],
    columns=["case"] + TRANSFER_FEATURES,
)

In [ ]:
## Double Check the Final Model

In [ ]:
edge_cases["probability"] = final_model.predict_proba(edge_cases[TRANSFER_FEATURES])[:, 1]
edge_cases["flagged"] = edge_cases["probability"] >= FINAL_THRESHOLD

In [ ]:
print(edge_cases[["case", "probability", "flagged"]].round(3).to_string(index=False))

                                case  probability  flagged
                      1 request, 404        0.000    False
 3 requests, all 404, distinct paths        0.025    False
20 requests, all 404, distinct paths        1.000     True
               20 requests, half 404        0.999     True
                      1 request, 200        0.025    False
     500 requests, all 200 (crawler)        0.000    False


In [ ]:
import os
import json
import joblib
import numpy as np

In [ ]:
MODEL_PATH = "recon_sensor_xgb.joblib"
META_PATH = "recon_sensor_xgb_metadata.json"


In [ ]:
joblib.dump(final_model, MODEL_PATH)

['recon_sensor_xgb.joblib']

In [ ]:
for stale in ["recon_sensor_xgb.json"]:
    if os.path.exists(stale):
        os.remove(stale)


In [ ]:
## Save Final Model MEtadata with real scripts and Limitations

In [ ]:
metadata = {
    "model": "recon_sensor_xgb",
    "format": "joblib",
    "role": "web directory brute-force and scan detector: windows where most requests return 404 across many distinct paths (MITRE T1595), not a general web attack classifier",
    "classes": {"0": "Normal", "1": "Scan_Burst"},
    "input": "per-IP window of Apache access log requests",
    "window_seconds": 10,
    "features": TRANSFER_FEATURES,
    "decision_threshold": FINAL_THRESHOLD,
    "versions": {
        "xgboost": xgboost.__version__,
        "numpy": np.__version__,
    },
    "training_data": "AIT Log Data Set v2.0 (Zenodo 5789064), intranet_server Apache access logs, 8 testbeds",
    "evaluation": "leave-one-testbed-out, out-of-fold probabilities",
    "metrics_at_threshold": {
        "attack_windows": 597,
        "benign_windows": 42427,
        "caught": 535,
        "false_positives": 0,
        "precision": 1.0,
        "recall": 0.8961,
        "roc_auc": 0.9832,
    },
    "feature_importance": {
        "ratio_404": 0.8705,
        "unique_paths": 0.0727,
        "req_count": 0.0440,
        "ratio_2xx": 0.0068,
        "ua_count": 0.0049,
        "unique_path_ratio": 0.0011,
    },
    "behavior_probes": {
        "1 request, 404": 0.000,
        "3 requests, all 404, distinct paths": 0.025,
        "20 requests, all 404, distinct paths": 1.000,
        "20 requests, half 404": 0.999,
        "1 request, 200": 0.025,
        "500 requests, all 200 (crawler)": 0.000,
    },
    "baseline_rule": {"rule": "req_count >= 100 in 10 s", "caught": 523, "false_positives": 0},
    "limitations": [
        "Lab data with simulated users, single WordPress site; largest benign burst was 96 requests per window",
        "Threshold chosen on the same data used for evaluation, treat metrics as an upper bound",
        "Low-volume attack steps such as webshell commands are mostly missed at this threshold",
        "A window of 20 requests with half returning 404 is flagged, a page with many broken assets could trigger it",
        "Validate on production Apache logs in log-only mode before feeding the correlator",
    ],
}


In [ ]:
with open(META_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

In [ ]:
loaded = joblib.load(MODEL_PATH)
X_check = win_df[TRANSFER_FEATURES]

In [ ]:
same = np.allclose(
    loaded.predict_proba(X_check)[:, 1],
    final_model.predict_proba(X_check)[:, 1],
)

In [ ]:
## Save and Download final Model

In [ ]:
print("Reload matches original predictions:", same)
print(f"Model size: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")

Reload matches original predictions: True
Model size: 198.5 KB
